# 드롭아웃



## 드롭아웃(Dropout) 이란?

드롭아웃은 학습 중 일부 뉴런 출력을 무작위로 0으로 만들어, 특정 뉴런에 과도하게 의존하는 현상을 줄이는 정규화(regularization) 기법입니다.

- 학습 중: 일부 뉴런을 확률적으로 끔
- 평가/추론 중: 뉴런을 끄지 않음

## 수식

학습 시(학습 모드):

- 마스크 샘플링: $m_i \sim \text{Bernoulli}(1-p)$
- 출력: $\tilde{h}_i = \frac{m_i}{1-p} h_i$

기호 설명:

- $h_i$: 드롭아웃 적용 전 활성값
- $\tilde{h}_i$: 드롭아웃 적용 후 활성값
- $p$: 드롭 확률(drop probability)
- $1-p$: 유지 확률(keep probability)

PyTorch는 Inverted Dropout을 사용하므로 학습 시 $\frac{1}{1-p}$로 스케일링하여,
평가 모드에서는 별도 스케일 조정 없이 그대로 사용합니다.

## 장점

- 과적합 완화(일반화 성능 개선)
- 앙상블과 유사한 효과(여러 부분 네트워크를 학습하는 효과)
- 구현이 간단하고 대부분 모델에 쉽게 적용 가능

## 단점

- 학습 속도가 다소 느려질 수 있음
- 드롭 비율(`p`) 튜닝이 필요함
- 데이터가 매우 작거나 모델이 작을 때는 성능 저하 가능

## 실무 팁

- 보통 `p=0.1 ~ 0.5` 범위에서 시작
- CNN에서는 완전연결층(FC) 쪽에 주로 사용
- 추론 시에는 반드시 `model.eval()`로 전환해야 함

In [1]:
import torch
import torch.nn as nn

# 재현성 확보
torch.manual_seed(42)

# 간단한 예제 입력 (배치 2개, 특성 5개)
x = torch.ones(2, 5)

drop = nn.Dropout(p=0.5)

# 1) 학습 모드: 일부 원소가 무작위로 0이 되고, 살아남은 값은 1/(1-p)=2배 스케일
# (입력이 전부 1이므로 출력에서 0 또는 2를 쉽게 관찰 가능)
drop.train()
out_train_1 = drop(x)
out_train_2 = drop(x)

# 2) 평가 모드: 드롭아웃 비활성화, 입력이 그대로 출력
drop.eval()
out_eval = drop(x)

print("입력 x:\n", x)
print("\n[train] 1회 출력:\n", out_train_1)
print("\n[train] 2회 출력(무작위성 확인):\n", out_train_2)
print("\n[eval] 출력(드롭아웃 비활성):\n", out_eval)

입력 x:
 tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

[train] 1회 출력:
 tensor([[2., 2., 2., 2., 0.],
        [2., 0., 0., 2., 2.]])

[train] 2회 출력(무작위성 확인):
 tensor([[2., 2., 0., 0., 2.],
        [0., 2., 0., 0., 2.]])

[eval] 출력(드롭아웃 비활성):
 tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])


In [2]:
# 드랍아웃 객체 생성, 드랍아웃 비율을 50%로 설정
dropout = nn.Dropout(p=0.5)

# 입력 데이터 텐서 생성
input_data = torch.randn(1, 6)  # 1x6 크기의 랜덤 텐서

# 드랍아웃 적용
output = dropout(input_data)

print("입력 데이터:\n", input_data)
print("\n드랍아웃 적용 후:\n", output)

입력 데이터:
 tensor([[ 1.3525,  0.6863, -0.3278,  0.7950,  0.2815,  0.0562]])

드랍아웃 적용 후:
 tensor([[ 0.0000,  1.3726, -0.6555,  1.5899,  0.5630,  0.1123]])
